# Create CAPES International Cooperation Awards

Creates OpenAlex award rows from CAPES' official Dados Abertos CKAN package for `Projetos de cooperacao internacional/pesquisa - SCBA`. The source package describes DRI international cooperation/research cost-support values granted to beneficiaries, with program, institution, graduate-program, date, and BRL amount fields.

**Awarding body:** Coordenacao de Aperfeicoamento de Pessoal de Nivel Superior (CAPES), `F4320321091`, DOI `10.13039/501100002322`.

**Source method:** method-1 CKAN open-data API + official CSV resources from `dadosabertos.capes.gov.br`.

**Schema choices:**
- One row per funded, deduped source grant/support row. Rows with non-positive `VL_CONCEDIDO` are excluded because they do not represent funded awards. The source does not expose a universal native award id, so `funder_award_id` is a deterministic `capes-cooperation-{sha1}` over normalized identifying source fields.
- `amount` = official `VL_CONCEDIDO`, `currency = 'BRL'`; CAPES metadata defines `VL_CONCEDIDO`, `VL_EMPENHADO`, and `VL_PAGO` as values in Brazilian reais.
- `lead_investigator` = source beneficiary; affiliation is the CAPES institution.
- `funder_scheme` = `SG_PROGRAMA_FOMENTO` (e.g., CAPES-PRINT, MOVE, COFECUB).
- `funding_type = 'research'` because the package is specifically international cooperation/research support.

Contractor has no S3/Databricks access; admin must upload the parquet, run this notebook, inspect verification cells, and run the aggregate CreateAwards notebook later.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.capes_cooperation_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/capes_cooperation/capes_cooperation_projects.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 2,065 funded rows.
SELECT COUNT(*) AS total_capes_cooperation_rows
FROM openalex.awards.capes_cooperation_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.capes_cooperation_raw;

In [ ]:
%sql
-- Sample raw CAPES cooperation data.
SELECT
    funder_award_id,
    display_name,
    beneficiary_name,
    funder_scheme,
    institution_name,
    graduate_program_name,
    amount,
    currency,
    source_year,
    start_date,
    end_date,
    landing_page_url
FROM openalex.awards.capes_cooperation_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'capes_cooperation_raw'
  AND LOWER(column_name) RLIKE 'amount|currency|valor|vl_|paid|committed';

In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(currency) AS rows_with_currency,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 1) AS pct_currency,
    COUNT(start_date) AS rows_with_start_date,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    COUNT(end_date) AS rows_with_end_date,
    ROUND(try_divide(COUNT(end_date), COUNT(*)) * 100.0, 1) AS pct_end_date,
    MIN(TRY_CAST(source_year AS INT)) AS min_source_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_source_year,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_brl
FROM openalex.awards.capes_cooperation_raw;

In [ ]:
%sql
-- Native-key inspection. funder_award_id is deterministic from normalized source fields.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_row_hash) AS distinct_source_row_hashes
FROM openalex.awards.capes_cooperation_raw;

In [ ]:
%sql
-- Must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.capes_cooperation_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY rows DESC, funder_award_id;

In [ ]:
%sql
-- Program/scheme distribution and money shape.
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_brl
FROM openalex.awards.capes_cooperation_raw
GROUP BY funder_scheme
ORDER BY rows DESC, funder_scheme;

In [ ]:
%sql
-- Year distribution.
SELECT
    TRY_CAST(source_year AS INT) AS source_year,
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_brl
FROM openalex.awards.capes_cooperation_raw
GROUP BY TRY_CAST(source_year AS INT)
ORDER BY source_year;

In [ ]:
%sql
-- Field coverage for beneficiary/institution/program metadata.
SELECT
    COUNT(*) AS total,
    COUNT(beneficiary_name) AS has_beneficiary,
    ROUND(try_divide(COUNT(beneficiary_name), COUNT(*)) * 100.0, 1) AS pct_beneficiary,
    COUNT(institution_name) AS has_institution,
    ROUND(try_divide(COUNT(institution_name), COUNT(*)) * 100.0, 1) AS pct_institution,
    COUNT(graduate_program_name) AS has_graduate_program,
    ROUND(try_divide(COUNT(graduate_program_name), COUNT(*)) * 100.0, 1) AS pct_graduate_program,
    COUNT(withdrawal_date) AS rows_with_withdrawal_date
FROM openalex.awards.capes_cooperation_raw;

## Step 1.6: Fail-fast - verify CAPES funder row exists

The Step 2 transform cross joins against `openalex.common.funder`. If this funder row is absent, the transform would silently emit zero awards. This assertion must pass before Step 2 runs.

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for CAPES F4320321091'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320321091;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.capes_cooperation_awards
USING delta
AS
WITH
capes_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321091
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 'BRL' ELSE NULL END AS parsed_currency,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_source_year,
        TRY_CAST(source_end_year AS INT) AS parsed_end_year
    FROM openalex.awards.capes_cooperation_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS id,

        TRIM(g.display_name) AS display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END AS description,

        f.funder_id,
        g.native_award_id AS funder_award_id,

        g.parsed_amount AS amount,
        g.parsed_currency AS currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,

        'research' AS funding_type,

        COALESCE(NULLIF(TRIM(g.funder_scheme), ''), 'International cooperation') AS funder_scheme,

        'capes_cooperacao_internacional' AS provenance,

        g.parsed_start_date AS start_date,
        g.parsed_end_date AS end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_source_year) AS start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_end_year) AS end_year,

        CASE
            WHEN g.beneficiary_name IS NULL OR TRIM(g.beneficiary_name) = '' THEN NULL
            ELSE struct(
                NULLIF(TRIM(g.beneficiary_given_name), '') AS given_name,
                NULLIF(TRIM(g.beneficiary_family_name), '') AS family_name,
                CAST(NULL AS STRING) AS orcid,
                g.parsed_start_date AS role_start,
                struct(
                    NULLIF(TRIM(g.institution_name), '') AS name,
                    'BR' AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        END AS lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,

        NULLIF(TRIM(g.landing_page_url), '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,

        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) AS works_api_url,

        current_timestamp() AS created_date,
        current_timestamp() AS updated_date

    FROM raw_prepared g
    CROSS JOIN capes_funder f
)

SELECT * FROM awards_transformed;

In [ ]:
%sql
SELECT COUNT(*) AS transformed_rows
FROM openalex.awards.capes_cooperation_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.capes_cooperation_awards;

## Step 3: Delete Existing Rows and Insert Fresh CAPES Cooperation Awards

In [ ]:
%sql
-- Remove previous CAPES cooperation data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'capes_cooperacao_internacional' AND priority = 144;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    144 as priority
FROM openalex.awards.capes_cooperation_awards;

## Step 6: Verification

In [ ]:
%sql
-- Confirm rows inserted at the expected provenance/priority.
SELECT
    provenance,
    priority,
    COUNT(*) AS rows,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'capes_cooperacao_internacional'
  AND priority = 144
GROUP BY provenance, priority;

In [ ]:
%sql
-- Must return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'capes_cooperacao_internacional'
  AND priority = 144
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY rows DESC, id;

In [ ]:
%sql
-- Amount/currency verification. This ingest should not need a Section 6.7 waiver.
SELECT
    currency,
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    MIN(amount) AS min_amount,
    PERCENTILE_APPROX(amount, 0.5) AS median_amount,
    MAX(amount) AS max_amount,
    ROUND(SUM(amount), 2) AS total_amount
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'capes_cooperacao_internacional'
  AND priority = 144
GROUP BY currency;

In [ ]:
%sql
-- Program/scheme distribution after transform.
SELECT funder_scheme, COUNT(*) AS rows, ROUND(SUM(amount), 2) AS total_brl
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'capes_cooperacao_internacional'
  AND priority = 144
GROUP BY funder_scheme
ORDER BY rows DESC, funder_scheme;

In [ ]:
%sql
-- Date/year verification.
SELECT
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year,
    MIN(end_year) AS min_end_year,
    MAX(end_year) AS max_end_year,
    COUNT(start_date) AS rows_with_start_date,
    COUNT(end_date) AS rows_with_end_date
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'capes_cooperacao_internacional'
  AND priority = 144;

In [ ]:
%sql
-- Sample transformed award rows.
SELECT
    id,
    display_name,
    funder_award_id,
    amount,
    currency,
    funding_type,
    funder_scheme,
    start_year,
    end_year,
    lead_investigator,
    landing_page_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'capes_cooperacao_internacional'
  AND priority = 144
ORDER BY start_year, funder_scheme, display_name
LIMIT 20;

## Handoff / Admin Notes

- Contractor-complete only: source script, notebook, CreateAwards registry entry, tracker row, and local validation are included in the PR.
- Admin must upload `capes_cooperation_projects.parquet` to `s3://openalex-ingest/awards/capes_cooperation/capes_cooperation_projects.parquet`, run this notebook on Databricks, inspect Step 6 outputs, then run the aggregate CreateAwards notebook.
- Do not add job YAML until the Databricks run and QA have succeeded.